In [13]:
import time
import requests

BASE_URL = "https://data.stats.gov.cn/dg/website/publicrelease/en/web/external"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Accept": "application/json, text/javascript, */*",
    "Referer": "https://data.stats.gov.cn/",
    "Content-Type": "application/json",
}
REQUEST_DELAY = 1.0

session = requests.Session()
session.headers.update(HEADERS)


# ─────────────────────────────────────────────
# STEP 1: TREE — get all leaf catalog nodes
# ─────────────────────────────────────────────

def get_tree(pid: str, code: int = 4) -> list:
    url = f"{BASE_URL}/new/queryIndexTreeAsync"
    res = session.get(url, params={"pid": pid, "code": code})
    res.raise_for_status()
    time.sleep(REQUEST_DELAY)
    return res.json().get("data", [])


def get_leaf_catalogs(pid: str) -> list[dict]:
    """Recursively walk tree until isLeaf=True."""
    nodes = get_tree(pid)
    leaves = []
    for node in nodes:
        if node.get("isLeaf"):
            leaves.append({
                "id":   node["_id"],
                "name": node["name"],
                "pid":  node["treeinfo_pid"],
            })
        else:
            leaves.extend(get_leaf_catalogs(node["_id"]))
    return leaves


# ─────────────────────────────────────────────
# STEP 2+3: REGIONS — get all provinces
# ─────────────────────────────────────────────

def get_region_catalog_id(indicator_cid: str) -> str:
    """
    getDaCatalogTreeByIndicatorCid → find the 'National Total' catalog id
    which contains all provinces.
    """
    url = f"{BASE_URL}/getDaCatalogTreeByIndicatorCid"
    res = session.get(url, params={"indicatorCid": indicator_cid})
    res.raise_for_status()
    time.sleep(REQUEST_DELAY)

    data = res.json().get("data", [])
    print(data)
    for item in data:
        # 'National Total' is level 1, no children — use it for all provinces
        if "children" not in item:
            return item["_id"]
    # fallback: return first item
    return data[0]["_id"] if data else ""


def get_regions(da_cid: str) -> list[dict]:
    """getDasByDaCatalogId → list of {text, value} for all provinces."""
    url = f"{BASE_URL}/getDasByDaCatalogId"
    res = session.get(url, params={"daCid": da_cid})
    res.raise_for_status()
    time.sleep(REQUEST_DELAY)

    return [
        {"text": item["show_name"], "value": item["name_value"]}
        for item in res.json().get("data", [])
    ]


# ─────────────────────────────────────────────
# STEP 4: INDICATORS — get indicator IDs for a catalog
# ─────────────────────────────────────────────

def get_indicators(cid: str, dt: str = "2025-2026") -> list[dict]:
    """
    queryIndicatorsByCid → list of indicators with their _id fields.
    Returns: [{"id": "...", "name": "...", "unit": "..."}]
    """
    url = f"{BASE_URL}/new/queryIndicatorsByCid"
    res = session.get(url, params={"cid": cid, "dt": dt, "name": ""})
    res.raise_for_status()
    time.sleep(REQUEST_DELAY)

    items = res.json().get("data", {}).get("list", [])
    return [
        {
            "id":   item["_id"],
            "name": item["i_showname"].strip(),
            "unit": item.get("du_name", ""),
        }
        for item in items
    ]


# ─────────────────────────────────────────────
# STEP 5: FETCH DATA
# ─────────────────────────────────────────────

def fetch_data(
    cid: str,
    indicator_ids: list[str],
    regions: list[dict],
    dts: list[str],
    root_id: str = "f4c6cd795fea436c807163397dd36b98",
) -> dict:
    """
    POST getEsDataByCidAndDt → actual time-series data.

    dts format: ["202403MM-202602MM"]  (YYYYMMTT range, TT=MM for monthly)
    """
    url = f"{BASE_URL}/getEsDataByCidAndDt"
    payload = {
        "cid":          cid,
        "indicatorIds": indicator_ids,
        "daCatalogId":  "",
        "das":          regions,
        "showType":     3,
        "dts":          dts,
        "rootId":       root_id,
    }
    res = session.post(url, json=payload)
    res.raise_for_status()
    time.sleep(REQUEST_DELAY)
    return res.json()


# ─────────────────────────────────────────────
# Single Energy Recode
# ─────────────────────────────────────────────

def single_energy_record(cid):
    da_cid     = get_region_catalog_id(cid)
    regions    = get_regions(da_cid)
    indicators = get_indicators(cid)
    raw        = fetch_data(cid, [indicators[0]["id"]], regions, ["202503MM-202602MM"])    
    return raw


# ─────────────────────────────────────────────
# FULL CRAWL
# ─────────────────────────────────────────────

def crawl(
    root_pid: str = "76e04d7533764d4384b0cd8d71deccbe",
    root_id:  str = "f4c6cd795fea436c807163397dd36b98",
    dt_range: str = "202403MM-202602MM",
    indicator_dt: str = "2025-2026",
):
    print("[1/4] Discovering leaf catalogs...")
    leaves = get_leaf_catalogs(root_pid)
    print(f"      Found {len(leaves)} leaf catalogs")

    all_results = {}

    for i, leaf in enumerate(leaves, 1):
        cid  = leaf["id"]
        name = leaf["name"]
        print(f"\n[{i}/{len(leaves)}] {name} (cid={cid})")

        # Step 2+3: regions
        print("      → fetching regions...")
        da_cid  = get_region_catalog_id(cid)
        regions = get_regions(da_cid)
        print(f"         {len(regions)} regions")

        # Step 4: indicators
        print("      → fetching indicators...")
        indicators = get_indicators(cid, dt=indicator_dt)
        print(f"         {len(indicators)} indicators: {[x['name'] for x in indicators]}")

        if not indicators:
            print("         ⚠ no indicators found, skipping")
            continue

        indicator_ids = [x["id"] for x in indicators]

        # Step 5: data
        print(f"      → fetching data ({dt_range})...")
        try:
            raw = fetch_data(
                cid=cid,
                indicator_ids=indicator_ids,
                regions=regions,
                dts=[dt_range],
                root_id=root_id,
            )
            all_results[name] = {
                "cid":        cid,
                "indicators": indicators,
                "regions":    regions,
                "raw":        raw,
            }
            print(f"         ✓ data fetched")
        except Exception as e:
            print(f"         ✗ Error: {e}")

    return all_results



In [14]:


ENERGY_TYPES = [
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Coal",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.9a59860aa4a9442e91d9aa01762e3284",
        "name": "Coal",
        "_id": "9a59860aa4a9442e91d9aa01762e3284",
        "publicrelease_web_catalog_id": "9a59860aa4a9442e91d9aa01762e3284",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Crude oil",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.ed34745091424311b9d62bbc16c1ea3b",
        "name": "Crude oil",
        "_id": "ed34745091424311b9d62bbc16c1ea3b",
        "publicrelease_web_catalog_id": "ed34745091424311b9d62bbc16c1ea3b",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Natural Gas",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.e32908a742c14e46a1525ab9549d9d6a",
        "name": "Natural Gas",
        "_id": "e32908a742c14e46a1525ab9549d9d6a",
        "publicrelease_web_catalog_id": "e32908a742c14e46a1525ab9549d9d6a",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Coalbed Gas",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.9d550c7d47144421b950ece73418c0a5",
        "name": "Coalbed Gas",
        "_id": "9d550c7d47144421b950ece73418c0a5",
        "publicrelease_web_catalog_id": "9d550c7d47144421b950ece73418c0a5",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "LNG",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.1a740c9519b54bcaac1db54804415356",
        "name": "LNG",
        "_id": "1a740c9519b54bcaac1db54804415356",
        "publicrelease_web_catalog_id": "1a740c9519b54bcaac1db54804415356",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Processing Volume of Crude oil",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.7da537ad56414da9a33427d3da95e434",
        "name": "Processing Volume of Crude oil",
        "_id": "7da537ad56414da9a33427d3da95e434",
        "publicrelease_web_catalog_id": "7da537ad56414da9a33427d3da95e434",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Gasoline",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.4e5c1ce4ff36454a8a2358c3c6b049d4",
        "name": "Gasoline",
        "_id": "4e5c1ce4ff36454a8a2358c3c6b049d4",
        "publicrelease_web_catalog_id": "4e5c1ce4ff36454a8a2358c3c6b049d4",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Kerosene",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.50d9afcb5bce42cb8def855131785cbe",
        "name": "Kerosene",
        "_id": "50d9afcb5bce42cb8def855131785cbe",
        "publicrelease_web_catalog_id": "50d9afcb5bce42cb8def855131785cbe",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Diesel Oil",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.534e11f26f95480582fe7b15acd09b03",
        "name": "Diesel Oil",
        "_id": "534e11f26f95480582fe7b15acd09b03",
        "publicrelease_web_catalog_id": "534e11f26f95480582fe7b15acd09b03",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Fuel Oil",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.1541c7cffca64ed1865643f76e84e429",
        "name": "Fuel Oil",
        "_id": "1541c7cffca64ed1865643f76e84e429",
        "publicrelease_web_catalog_id": "1541c7cffca64ed1865643f76e84e429",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Naphtha",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.14104b0fe87c4cce955f03bba283cbfd",
        "name": "Naphtha",
        "_id": "14104b0fe87c4cce955f03bba283cbfd",
        "publicrelease_web_catalog_id": "14104b0fe87c4cce955f03bba283cbfd",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "LPG",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.aff1ae7bcd2e4c90a11f3dd1965bce9e",
        "name": "LPG",
        "_id": "aff1ae7bcd2e4c90a11f3dd1965bce9e",
        "publicrelease_web_catalog_id": "aff1ae7bcd2e4c90a11f3dd1965bce9e",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Petroleum Coke",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.efd6dbad356b4928a5527b42e7c1093d",
        "name": "Petroleum Coke",
        "_id": "efd6dbad356b4928a5527b42e7c1093d",
        "publicrelease_web_catalog_id": "efd6dbad356b4928a5527b42e7c1093d",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Asphalt",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.336384a105dd4ed3975a4334c91c4b8c",
        "name": "Asphalt",
        "_id": "336384a105dd4ed3975a4334c91c4b8c",
        "publicrelease_web_catalog_id": "336384a105dd4ed3975a4334c91c4b8c",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Coke",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.0e83c61d3fe642b28b6294afcea3039b",
        "name": "Coke",
        "_id": "0e83c61d3fe642b28b6294afcea3039b",
        "publicrelease_web_catalog_id": "0e83c61d3fe642b28b6294afcea3039b",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Output of Electricity",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.f76c5af9a1604d1b906463b208bdd675",
        "name": "Output of Electricity",
        "_id": "f76c5af9a1604d1b906463b208bdd675",
        "publicrelease_web_catalog_id": "f76c5af9a1604d1b906463b208bdd675",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Thermal Power",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.c81899a0508d4fdc9ad5786ef7810cca",
        "name": "Thermal Power",
        "_id": "c81899a0508d4fdc9ad5786ef7810cca",
        "publicrelease_web_catalog_id": "c81899a0508d4fdc9ad5786ef7810cca",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Hydro-electric Power",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.5a58168195604a348e6a5d909552187e",
        "name": "Hydro-electric Power",
        "_id": "5a58168195604a348e6a5d909552187e",
        "publicrelease_web_catalog_id": "5a58168195604a348e6a5d909552187e",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Nuclear Power",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.c8afa9ab62994303932ec9eee0b8833a",
        "name": "Nuclear Power",
        "_id": "c8afa9ab62994303932ec9eee0b8833a",
        "publicrelease_web_catalog_id": "c8afa9ab62994303932ec9eee0b8833a",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Wind Power",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.35d85b5392be4ce19cae8d6ae83932aa",
        "name": "Wind Power",
        "_id": "35d85b5392be4ce19cae8d6ae83932aa",
        "publicrelease_web_catalog_id": "35d85b5392be4ce19cae8d6ae83932aa",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Solar Power",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.0262c099768642d5a5dcbb5928f7c910",
        "name": "Solar Power",
        "_id": "0262c099768642d5a5dcbb5928f7c910",
        "publicrelease_web_catalog_id": "0262c099768642d5a5dcbb5928f7c910",
        "treeinfo_icon": None
    },
    {
        "explain": None,
        "sdate": "1998",
        "_name": "Gas",
        "treeinfo_pid": "f46bf43e25374f5b9e2181676d582356",
        "treeinfo_level": 6,
        "type": "catalog",
        "edate": None,
        "isLeaf": 1,
        "treeinfo_globalid": "69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.bb881d7174db4fcc81f52159c5f0e0af",
        "name": "Gas",
        "_id": "bb881d7174db4fcc81f52159c5f0e0af",
        "publicrelease_web_catalog_id": "bb881d7174db4fcc81f52159c5f0e0af",
        "treeinfo_icon": None
    }
]

In [32]:
import pandas as pd



def to_tabular_format(data, energy_type):
    raw_data = data

    rows = []

    for record in raw_data:
        base = {
            "code": record.get("code"),
            "name": record.get("name"),
            "energy_type": energy_type
        }

        for item in record.get("values", []):
            row = base.copy()
            row.update(item)   # merge all fields
            rows.append(row)

    return pd.DataFrame(rows)


def get_all_energy_type(name=None):
    if name:
        return next(
            (item for item in ENERGY_TYPES if item["name"] == name.lower(
            ) or item["_name"] == name),
            None
        )
    return ENERGY_TYPES




In [16]:
energy_type = get_all_energy_type("Coal")


In [18]:
energy_type

{'explain': None,
 'sdate': '1998',
 '_name': 'Coal',
 'treeinfo_pid': 'f46bf43e25374f5b9e2181676d582356',
 'treeinfo_level': 6,
 'type': 'catalog',
 'edate': None,
 'isLeaf': 1,
 'treeinfo_globalid': '69c574ab128a44e595cc0b24502b771b.6b8e8719075f43d4bf83b261208b37b5.f4c6cd795fea436c807163397dd36b98.76e04d7533764d4384b0cd8d71deccbe.f46bf43e25374f5b9e2181676d582356.9a59860aa4a9442e91d9aa01762e3284',
 'name': 'Coal',
 '_id': '9a59860aa4a9442e91d9aa01762e3284',
 'publicrelease_web_catalog_id': '9a59860aa4a9442e91d9aa01762e3284',
 'treeinfo_icon': None}

In [19]:
# data = single_energy_record(energy_type['id'])
cid = energy_type['_id']
da_cid     = get_region_catalog_id(cid)


[{'_name': 'National Total', 'treeinfo_globalid': '6a99abef22d44119bdf5f8e2451ac390.a10dceae75d245008bf4b9a0e6fe1d55', 'show_name': 'National Total', 'treeinfo_pid': '6a99abef22d44119bdf5f8e2451ac390', 'treeinfo_level': 2, 'name': '全部地区', '_level': 1, '_id': 'a10dceae75d245008bf4b9a0e6fe1d55', 'publicrelease_web_dacatalog_id': 'a10dceae75d245008bf4b9a0e6fe1d55'}, {'_name': 'Four Regions', 'treeinfo_globalid': '6a99abef22d44119bdf5f8e2451ac390.a40ae1698e08488b9ce67a2887330997', 'children': [{'_name': 'Eastern Region', 'treeinfo_globalid': '6a99abef22d44119bdf5f8e2451ac390.a40ae1698e08488b9ce67a2887330997.ea5f666a6d514569a29f8c4af7a26fa7', 'show_name': 'Eastern Region', 'treeinfo_pid': 'a40ae1698e08488b9ce67a2887330997', 'treeinfo_level': 3, 'name': '东部地区', '_level': 2, '_id': 'ea5f666a6d514569a29f8c4af7a26fa7', 'publicrelease_web_dacatalog_id': 'ea5f666a6d514569a29f8c4af7a26fa7'}, {'_name': 'The Intermediate Region', 'treeinfo_globalid': '6a99abef22d44119bdf5f8e2451ac390.a40ae1698e08488

In [20]:
da_cid

'a10dceae75d245008bf4b9a0e6fe1d55'

In [21]:
regions    = get_regions(da_cid)



In [22]:
regions

[{'text': 'Beijing', 'value': '110000000000'},
 {'text': 'Tianjin', 'value': '120000000000'},
 {'text': 'Hebei', 'value': '130000000000'},
 {'text': 'Shanxi', 'value': '140000000000'},
 {'text': 'Inner Mongolia', 'value': '150000000000'},
 {'text': 'Liaoning', 'value': '210000000000'},
 {'text': 'Jilin', 'value': '220000000000'},
 {'text': 'Heilongjiang', 'value': '230000000000'},
 {'text': 'Shanghai', 'value': '310000000000'},
 {'text': 'Jiangsu', 'value': '320000000000'},
 {'text': 'Zhejiang', 'value': '330000000000'},
 {'text': 'Anhui', 'value': '340000000000'},
 {'text': 'Fujian', 'value': '350000000000'},
 {'text': 'Jiangxi', 'value': '360000000000'},
 {'text': 'Shandong', 'value': '370000000000'},
 {'text': 'Henan', 'value': '410000000000'},
 {'text': 'Hubei', 'value': '420000000000'},
 {'text': 'Hunan', 'value': '430000000000'},
 {'text': 'Guangdong', 'value': '440000000000'},
 {'text': 'Guangxi', 'value': '450000000000'},
 {'text': 'Hainan', 'value': '460000000000'},
 {'text': 

In [23]:
indicators = get_indicators(cid)


In [24]:
raw        = fetch_data(cid, [indicators[0]["id"]], regions, ["202503MM-202602MM"])    


In [33]:
ss = to_tabular_format(raw['data'], energy_type["_name"])



In [36]:
print(raw['data'][0])

{'code': '202602MM', 'values': [{'i_showname': 'Output of Coal, Current Period (10000 tons) ', '_name': '能源生产量', 'ek_dp': '124740211529|295834ae23bc45ed8124d53634d35e60_1', 'kj1_name': '原煤', 'accuracy': '1', 'dp': '1', 'type': 'indicator', 'is_economy_chart_show_value': '1', 'ds_order': 1, 'du': 'ad72a4425b6f4299bb9e34d3a6d5087e', 'kj3_name': None, 'chart_style_text': None, 'ek_dp_name': '能源生产量_原煤_本期', 'chart_style_value': None, 'value': '', 'ek_name': '能源生产量_原煤', 'order': 1, 'area': 'Beijing', 'dp_name': '本期', 'daterange_sdate': '1998-01-01 00:00:00', 'is_economy_chart_show_text': '是', 'kj2_name': None, 'ek': '124740211529|295834ae23bc45ed8124d53634d35e60', 'i': '295834ae23bc45ed8124d53634d35e60', 'dead_line_value': None, 'areaCode': '110000000000', 'kj2': None, 'kj1': '124740211529', 'catalogid': '9a59860aa4a9442e91d9aa01762e3284', 'daterange_edate': None, 'kj3': None, 'i_mark': None, '_id': '00930ccfd5944d81a1f69ed38e7c2dfe', 'du_name': None}, {'i_showname': 'Output of Coal, Current

In [30]:
import json
# Define the filename
filename = "data_export.json"

# Writing to the file
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(raw, f, indent=4, ensure_ascii=False)

print(f"Successfully saved to {filename}")

Successfully saved to data_export.json


In [ ]:

df = to_tabular_format(raw['data'],energy_type["_name"])


       code      name energy_type  \
0  202602MM  Feb 2026        Coal   
1  202602MM  Feb 2026        Coal   

                                     i_showname  _name  \
0  Output of Coal, Current Period (10000 tons)   能源生产量   
1  Output of Coal, Current Period (10000 tons)   能源生产量   

                                             ek_dp kj1_name accuracy dp  \
0  124740211529|295834ae23bc45ed8124d53634d35e60_1       原煤        1  1   
1  124740211529|295834ae23bc45ed8124d53634d35e60_1       原煤        1  1   

        type  ... dead_line_value      areaCode   kj2           kj1  \
0  indicator  ...            None  110000000000  None  124740211529   
1  indicator  ...            None  120000000000  None  124740211529   

                          catalogid daterange_edate   kj3 i_mark  \
0  9a59860aa4a9442e91d9aa01762e3284            None  None   None   
1  9a59860aa4a9442e91d9aa01762e3284            None  None   None   

                                _id  du_name  
0  00930ccfd5944d81a1

In [43]:
df_filtered = df[['code', 'area', 'value']]

In [46]:
df_filtered.tail(40)

,code,area,value
332,202504MM,Sichuan,174.1
333,202504MM,Guizhou,1356.8
334,202504MM,Yunnan,694.8
335,202504MM,Xizang,0.0
336,202504MM,Shaanxi,6587.0
337,202504MM,Gansu,578.1
338,202504MM,Qinghai,58.6
339,202504MM,Ningxia,802.0
340,202504MM,Xinjiang,3923.9
341,202503MM,Beijing,0.0
